<a href="https://colab.research.google.com/github/senthil1256/SC1003-Intro-to-Computational-Thinking-Group-project/blob/main/Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction of Mini Assignment

This mini assignment aims to design and implement an automated team-formation application, organising NTU students into groups of fair and diverse teams within their respective tutorial groups by considering school affliation, gender balance and CGPA.


**Computational Thinking (CT)** – While working on this project, you and your team must reflect,
relate, and document how you have applied computational thinking concepts or processes
when solving this problem in Jupyter notebook. Each stage / process / action / decision must
be supported by the computational thinking principles with justification or thought processes.

We have decomposed the problem into several parts and have developed a solution that will comprise of 5 components.

1. Choose team size
2. Read csv and store data
3. Team formation function will balance the team size optimally
4. Greedy algorithm which will use the our native penalty score system to     form the best team
5. Output the graphs containing team diversity data into a word document, and the new CSV file


The **TutorialGroup Class** is defined here. This will be used by the algorithm later to access the data of students in the different tutorial groups

Each column name (e.g student_ID, name, ect.) of the original CSV file is given a list to store data in the **attributes**

**Mean_cgpa attribute** is created to compute and store tutorial group mean cgpa.
**Teams attribute list** is created to store the output teams from the alogorthim.

Purpose of the **add_data method** is to append each element of the row seperated by the commas, into the respective attribute lists.

The cgpa is converted from a string to a float, before getting appended into the list, to allow arithmetic computation later.


In [ ]:
import csv, random
import matplotlib.pyplot as plt
from docx import Document
from docx.shared import Inches
import os
import ipywidgets as widgets
from IPython.display import display

# DATA STRUCTURE FOR TUTORIAL GROUP (PARENT NODES)
class TutorialGroup:
    def __init__(self):
        self.studentID = []
        self.school = []
        self.name = []
        self.gender = []
        self.cgpa = []
        self.mean_cgpa = 0.0
        self.teams = []

    #To append the data from row to column format by 5 list
    def add_data(self, data):
        self.studentID.append(data[1])
        self.school.append(data[2])
        self.name.append(data[3])
        self.gender.append(data[4])
        self.cgpa.append(float(data[5]))


ModuleNotFoundError: No module named 'docx'

In [ ]:
!pip install python-docx

group_name variable is initiliased with an empty string at first. It is updated with the current row tutorial group when iterating through each row.

data_set dictionary is created. Keys will be the unique tutorial group names,Values will be the TutorialGroup object for that tutorial group.

CSV is opened in 'read' mode and using a For loop, the reader reads row by row

row[0] is the tutorial group name in the current row iteration and is compared with the value of group_name which is the group name in previous row.

If the tutorial group name is different and does not exist as a key yet, a new key for data_set dictionary is created and a TutorialGroup object is assigned as a value.

The attributes can be accessed as follows:
data_set["tutorial group name"].name of attribute

Next the mean cgpa for each tutorial group is calculated by summing up all the cgpa elements in the cgpa attribute list and dividing by length of list.

This value is rounded to 4 decimal places and assigned to the mean_cgpa attribute.

Lastly the CSV file is closed


In [ ]:
def CSV_reader():
  group_name = ''
  data_set = {}
  #1. Start to read, extract and store data from csv file
  print("Start to extract the data from the csv file and store in specific data structure...")
  with open("records.csv", 'r') as file:
      reader = csv.reader(file)
      next(reader) #Skip to the second row because first row is header
      for row in reader:
          #Create the Dict's key: Tutorial Group number and value: class TutorialGroup()
          if group_name != row[0]:
              group_name = row[0]
              if group_name not in data_set:
                data_set[group_name] = TutorialGroup()
          data_set[group_name].add_data(row)

      for group_name, data in data_set.items():
          data_set[group_name].mean_cgpa = \
              round(sum(data_set[group_name].cgpa) / len(data_set[group_name].cgpa), 4)
      file.close()

  return data_set

##create_team_formation Function
This function is used to create a balanced team size for all the teams if there are remainder.\

For example: \
**Team size: 6**\
**Tutorial group size: 50**\\

There will be a remainder 2. If remainder is not zero then we assume the will be one more team to balance the all the teams' size. \
Then we will create a list of team formation which is 9 teams of team size 6 [6, 6, 6, 6, 6, 6, 6, 6, 6]. \
After that for loop the ((team size x number teams) - total people) which is for loop 4 times. 4 teams will minus one member be like [5, 5, 5, 5, 6, 6, 6, 6, 6].\
Lastly, reverse the loop to ensure the larger team in front of list [6, 6, 6, 6, 6, 5, 5, 5, 5].


##Pseudocode for create_team_formation()
```
FUNCTION create_team_formation(INPUT total people in Tutorial Group)
	DECLARE global TEAM_SIZE
	FIND number of teams
	FIND remainder that cannot form a full team
	DECLARE team_formation list to empty list

	IF remainder is not zero
		PLUS one more team number
	END IF
	CREATE number teams with TEAM_SIZE in team_formation

	FOR i from range((TEAM_SIZE x number teams) - total people in tutorial group)
		MINUS one member from team_formation[i]
	END FOR

	SORT team_formation in reverse way
	RETURN team_formation

END FUNCTION

```

##Flowchart: create_team_formation function
```
```

In [ ]:
def create_team_formation(total_people):
    global TEAM_SIZE
    num_teams = total_people // TEAM_SIZE
    remainder = total_people % TEAM_SIZE
    team_formation = []

    if remainder != 0:
        num_teams += 1
    team_formation = [TEAM_SIZE] * (num_teams)

    for i in range((TEAM_SIZE * num_teams)-total_people):
        team_formation[i] -= 1
    team_formation.sort(reverse = True)

    return team_formation

#Penalty Score System
This function is how we determine the fairness of each group, abstracting away the fairness criteria from the greedy algorithm to make the sorting easier to program. Furthermore, the importance of each criteria in fairness is customizable through its individual components' weights, making it easier to adjust if needed.

<br/>

## Design Considerations
The penalty system is **normalised** (a score of 0-1). A way to think about it is representing the fairness of each criteria as a percentage, 0 being the most fair and 1 being the least fair. Through normalising, it ensures that regardless of the team size, the penalty remains an accurate representation, synergizing well with the user interactivity function.


Without normalising and just using integers, the fairness of the larger team sizes would be inflated.

put the 2 flowcharts on penalty here

##**Pseudocode for penalty score**
```
FUNCTION Penalty score(group_data(data of entire tutorial group), team_data(data of team to calculate score))

  DECLARE WEIGHT_SCHOOL TO 0.25
  DECLARE WEIGHT_GENDER TO 0.25
  DECLARE WEIGHT_GPA_MEAN TO 0.30
  DECLARE WEIGHT_GPA_VAR TO 0.20

  DECLARE an empty list called index_list
  FOR student_id in team_data:
    FIND index of students BY student_id in group_data.studentID
    APPEND index to index_list
  END FOR


  # School Penalty
  DECLARE team_school as an empty list
  FOR index in index_list:
    APPEND group_data.school[index] to team_school
  END FOR

  FIND number_schools BY LENGTH of SET(team_school)

  IF LENGTH(team_school) > 1:
    FIND penalty_school BY (LENGTH(team_school) - number_schools) / (LENGTH(team_school) - 1)
  ELSE:
    penalty_school = 0
  END IF

  # Gender Penalty
  DECLARE team_gender as an empty list
  FOR index in index_list:
    APPEND group_data.school[index] to team_school
  END FOR
  FIND number_males by SUM of ‘Male’ in team_gender
  FIND penalty_gender BY ABS(number_males/ LENGTH(team_gender) - 0.5) / 0.5

  # Mean CGPA Penalty
  DECLARE team_cgpa as an empty list
  FOR index in index_list:
    APPEND group_data.cgpa[index] TO team_cgpa
  END FOR

  FIND team_mean_cgpa BY SUM(team_cgpa) / LENGTH(team_cgpa)

  FIND max_deviation_possible BY MAX(
  (MAX(group_data.cgpa) - group_data.mean_cgpa),
  (group_data.mean_cgpa - MIN(group_data.cgpa))
  )

  FIND penalty_cgpa BY ABS(group_data.mean_cgpa - team_mean_cgpa) / max_deviation_possible

  #CGPA Variance penalty
  FIND max_variance BY ((MAX(group_data.cgpa) - MIN(group_data.cgpa)) / 2) SQUARED
  DECLARE sum_squared_diff as 0
  FOR cgpa in team_cgpa:
    ADD (cgpa - team_mean_cgpa) SQUARED to sum_squared_diff
  END FOR

  FIND team_variance BY sum_squared_diff / LENGTH(team_cgpa)
  FIND penalty_variance BY team_variance / max_variance

  FIND total_penalty BY (WEIGHT_SCHOOL * penalty_school) + \
  (WEIGHT_GENDER * penalty_gender) + \
  (WEIGHT_GPA_MEAN * penalty_cgpa) + \
  (WEIGHT_GPA_VAR * penalty_variance)

  RETURN total_penalty
END FUNCTION
```


In [ ]:
def calculate_penalty(group_data, team_data):
    # Weights determine the priority of each variable
    WEIGHT_SCHOOL = 0.25
    WEIGHT_GENDER = 0.25
    WEIGHT_GPA_MEAN = 0.25
    WEIGHT_GPA_VAR = 0.25

    '''
    index_list:
    Get the specific index number/ position in the list from student id to
    access other value in other lists
    '''
    index_list = [group_data.studentID.index(student_id) for student_id in team_data]

    # School Penalty: Penalizes multiple students from the same school.
    team_school =[group_data.school[index] for index in index_list]
    number_schools = len(set(team_school))
    if len(team_school) > 1:
        penalty_school = (len(team_school) - number_schools) / (len(team_school)-1)
    else:
        penalty_school = 0

    # Gender Penalty: Penalizes gender imbalance.
    team_gender = [group_data.gender[index] for index in index_list]
    number_males = sum(1 for gender in team_gender if gender == 'Male')
    penalty_gender = abs(number_males/len(team_gender) - 0.5) / 0.5

    # Mean CGPA Penalty: Penalizes team whose average CGPA deviates from overall average CGPA
    team_cgpa = [group_data.cgpa[index] for index in index_list]
    team_mean_cgpa = sum(team_cgpa) / len(team_cgpa)
    penalty_cgpa = abs(group_data.mean_cgpa - team_mean_cgpa) / max(max(group_data.cgpa) - group_data.mean_cgpa, group_data.mean_cgpa - min(group_data.cgpa))

    # CGPA Variance Penalty: Penalizes teams with high CGPA variance.
    max_variance = (((max(group_data.cgpa) - min(group_data.cgpa))/2) ** 2)
    penalty_variance = (sum((cgpa - team_mean_cgpa) ** 2 for cgpa in team_cgpa) / len(team_cgpa)) / max_variance

    # Total Penalty: Weighted sum of all penalties.
    total_penalty = (WEIGHT_SCHOOL * penalty_school + \
                    WEIGHT_GENDER * penalty_gender + \
                    WEIGHT_GPA_MEAN * penalty_cgpa + \
                    WEIGHT_GPA_VAR * penalty_variance)

    return total_penalty

#Greedy Algorithm
A greedy algorithm is an algorithm strategy that takes the best choice in each step of the current state, in the hope that the result is the best or optimal globally.

Why the Greedy Algorithm?
1. First of all, the greedy algorithm can control the time complexity and space complexity to be low
2. Execution efficiency is higher because you do not need to traverse all possible solutions
3. The memory usage is relatively small, making it suitable for processing large-scale data.

Although the greedy algorithm can not guarantee that all problems can be solved globally, in many practical applications, its high efficiency and acceptable quality of the solution make it the preferred algorithm.

put the 2 flowcharts on greedy

Pseudocode for greedy grouping

```
FUNCTION greedy grouping(INPUT data and team_formation)
           
    Create a copy of all student IDs
    Create an empty list to store all teams

    FOR i from 0 to length(team_formation)-1
          APPEND teams to the empty list
                  
    FOR team in teams
        IF length(team) < target_size and remaining_students is not empty
            APPEND the best student to the current team
            REMOVE the student from the remaining students
    END FOR
           
    CREATE an empty list to store teams and their penalty scores
    FOR team in teams
        APPEND the penalty scores to teams
    END FOR
END FUNCTION        
```
Pseudocode for find_best_candidate
```
FUNCTION find_best_candidate(remaining_students, current_team, data)
    
    SET best_score = infinity
    SET best_candidate = first student in remaining_students
    
    FOR each student IN remaining_students:
        CREATE temp_team = current_team + [student]
        CALCULATE penalty_score FOR temp_team
        
        IF penalty_score < best_score:
            SET best_score = penalty_score
            SET best_candidate = student
    
    RETURN best_candidate
```

In [ ]:
def greedy_grouping(data, team_formation):
    remaining_students = data.studentID.copy()
    teams = []

    for number_team in team_formation:
        teams.append([])

    while remaining_students:
        for team in teams:
            team_num = teams.index(team)
            target_size = team_formation[team_num]
            if len(team) < target_size and remaining_students:
                best_candidate = find_best_candidate(remaining_students, team, data)
                team.append(best_candidate)
                remaining_students.remove(best_candidate)
            teams[team_num] = team

    team_with_penalty_list = []
    for team in teams:
        penalty = calculate_penalty(data, team)
        team_with_penalty_list.append((team, penalty))

    return team_with_penalty_list

def find_best_candidate(remaining_students, current_team, data):
    # if not candidates:
    #     return None
    best_score = float('inf')
    best_candidate = remaining_students[0]

    for student in remaining_students:
        temp_team = current_team[:] + [student]
        score = calculate_penalty(data, temp_team)
        if score < best_score:
            best_score = score
            best_candidate = student

    return best_candidate

#print_outcome function
This function is used to print out the result during development og program in terminal.\
Hence, we dont need to take long time to generate the graph for developer to view the result.\


In [ ]:
def print_outcome(team_num, team, penalty, data):
    gender_list = [data.gender[data.studentID.index(member)] for member in team]
    school_list = [data.school[data.studentID.index(member)] for member in team]
    unique_schools = set(school_list)
    print("=" * 70)
    print(f'Team {team_num+1:^10}')
    print(f'Penalty: {penalty:.4f} \
          Mean CGPA: {round(sum([data.cgpa[data.studentID.index(member)] for member in team])/len(team),4)}\
          Gender: {gender_list.count("Male")}M/{gender_list.count("Female")}F')
    print(f'SCH: {",".join([f"{sch:}({school_list.count(sch)})" for sch in unique_schools]):<45}')
    print(f"|{'StudentID':^10}|{'SCH':^9}|{'Name':^30}|{'Gender':^7}|{'CGPA':^5}|")
    print("=" * 70)
    for member in team:
        index = data.studentID.index(member)
        print(f"|{member:^10}|{data.school[index]:^9}|{data.name[index]:^30}|{data.gender[index]:^7}|{data.cgpa[index]:.2f}|")
    print("=" * 70)
    print()

##progressBar Function
This function has one progress bar only.\
It used to show the progress of the generating graph.\
Then user can know when the program is completely finish so that user wont disrupt the program until it crush.

##Pseudocode for progressBar
```
FUNCTION progressBar (INPUT data_set DICTIONARY)
	FIND the total groups inside the data_set

	CREATE progress bar = FloatProgress bar widget
		WITH value = 0.0,
		AND min = 0.0,
		AND max = total groups number,
		AND description = “Generating graph”,
		AND bar_style = “info”,
		AND style = {‘bar color’: ‘#2196F3},
		AND orientation = ‘horizontal’

	DISPLAY progress bar
	RETURN progress bar
END FUNCTION

```

In [ ]:
def progressBar(data_set):
  total_groups = len(data_set)
  progress_bar = widgets.FloatProgress(
      value=0.0,
      min=0.0,
      max=total_groups,
      description='Generating Charts:',
      bar_style='info',
      style={'bar_color': '#2196F3'},
      orientation='horizontal'
  )

  display(progress_bar)
  return progress_bar

#Creating the CSV, Word Document, and the graphs

Before generating the graph, the word document is created with doc = Document().
We loop through each tutorial group in the data_set. In the loop we create the list to store the info later used for the graph. For the schools we had to create another dictionary to store how many people from a certain team are in that school as a key. E.g {"NBS":[2,1,1]}

List comprehension was used to get the needed data before finding the CGPA mean, gender count and school variance.

Once we got all the nesccesary data, we can start plotting the graphs.
Each Tutorial Group's graph will be saved as a png and exported into the Word Document.

Also once all the teams are formed, it will be exported to team_allocation. With the headers being Team, Name, StudentID, CGPA, Gender, and School.

#Pseudocode for CSV, Word Doc, and Graph

```FUNCTION output_csv_word(data_set):

    CREATE a new Word document
    ADD heading "Team Data" to the document
    PRINT "Start to generate the charts for better result's view...")

    CREATE a progress bar for the number of tutorial groups
    SET group_processed = 0

    FOR each tutorial group in data_set:
        GET the list of teams in the group
        CREATE labels for each team (Team 1, Team 2,.....)

        initialized empty lists:
        cgpa_means
        male_counts
        female_counts
        GET all unique school names in the group
        CREATE dictionary school_counts with empty lists for each school

        FOR each team in the group:
            Calculate average CGPA for the team
            Append to cgpa_means

            Count number of males and females in the team
            Append to male_counts and female_counts

            Count number of students from each school in the team
            Append counts to corresponding school_counts lists

        CREATE 3-row subplot figure:
            row 1: Bar chart of cgpa_means per team + reference line for group average CGPA
            row 2: Bar chart of male_counts and female_counts per team
            row 3: Stacked bar chart of school_counts per team

        save the figure as png in "Graphs.png" folder

        ADD heading for this tutorial group in Word doc
        INSERT the saved chart image
        ADD a blank paragraph for spacing

        save the word document
        UPDATE progress bar

FUNCTION TeamSummary_csv(tutorial_group, team_number, team, data, csv_writer):
    FOR each student in team:
          GET student info (name, ID, gender, CGPA, school)
          WRITE a row to CSV

PRINT "WORD Report (FCE1_Team3_YongDingAn.csv.docx) Completed"
PRINT "Start to generate the CSV file.."

 OPEN CSV file for writing
    WRITE CSV header row

    FOR each tutorial group in data_set:
        SET team_counter = 1

        FOR each team in the group:
            CALL TeamSummary_csv to write all students in the team
            INCREMENT team_counter

PRINT "CSV file (FCE1_Team3_YongDingAn.csv.csv) Completed"
PRINT "Program Finished!!! Pls view the result in the WORD doc and the CSV file."```



In [ ]:
def output_csv_word(data_set):
  doc = Document()
  doc.add_heading("Team Data", level = 1)
  print("Start to generate the charts for better result's view...")
  progress_bar = progressBar(data_set)
  group_processed = 0

  for group_name, data in data_set.items():
      teams = data.teams  # list of (team_members, penalty)
      team_labels = [f"Team {i+1}" for i in range(len(teams))]

      cgpa_means = []
      male_counts = []
      female_counts = []
      school_names = sorted(set(data.school))
      school_counts = {school: [] for school in school_names}

      for team, penalty in teams:
          cgpas = [data.cgpa[data.studentID.index(s)] for s in team]
          cgpa_means.append(sum(cgpas) / len(cgpas))

          genders = [data.gender[data.studentID.index(s)] for s in team]
          male_counts.append(genders.count("Male"))
          female_counts.append(genders.count("Female"))

          team_schools = [data.school[data.studentID.index(s)] for s in team]
          for school in school_names:
              school_counts[school].append(team_schools.count(school))

      x = range(len(teams))
      fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
      fig.suptitle(f"Tutorial Group: {group_name}", fontsize=14, fontweight='bold')

      #CGPA Distribution (per team)
      axes[0].bar(x, cgpa_means, color='teal', alpha=0.7, label="Team Average CGPA")
      #Add reference line for tutorial group overall mean CGPA
      axes[0].axhline(data.mean_cgpa, color='red', linestyle='--', linewidth=2, label=f"Group Avg CGPA ({data.mean_cgpa:.2f})")
      axes[0].set_title("Average CGPA per Team")
      axes[0].set_ylabel("CGPA")
      axes[0].legend()

      #Gender Distribution (per team)
      axes[1].bar(x, male_counts, width=0.4, label='Male', color='skyblue')
      axes[1].bar([i + 0.4 for i in x], female_counts, width=0.4, label='Female', color='pink')
      axes[1].set_title("Gender Distribution per Team")
      axes[1].set_ylabel("Count")
      axes[1].legend()

      #School Distribution (stacked bar per team)
      bottom = [0]*len(teams)
      for school in school_names:
          axes[2].bar(x, school_counts[school], bottom=bottom, label=school)
          bottom = [bottom[i] + school_counts[school][i] for i in range(len(teams))]

      axes[2].set_title("School Distribution per Team")
      axes[2].set_ylabel("Count")
      axes[2].set_xticks([i + 0.2 for i in x])
      axes[2].set_xticklabels(team_labels, rotation=45, ha='right')
      axes[2].legend(title="School", bbox_to_anchor=(1, 1))

      plt.tight_layout(rect=[0, 0, 1, 0.95])
      chart_folder = "Graphs png"
      os.makedirs(chart_folder, exist_ok=True)
      img_path = os.path.join(chart_folder, f"{group_name}_chart.png")
      plt.savefig(img_path, bbox_inches= 'tight')
      plt.close()



      doc.add_heading(f"Tutorial Group: {group_name}", level =2)
      doc.add_picture(img_path, width=Inches(6))
      doc.add_paragraph("")

      output_path = "FCE1_Team3_YongDingAn.csv.docx"
      doc.save(output_path)

      group_processed += 1
      progress_bar.value = group_processed

  def TeamSummary_csv(tutorial_group, team_number, team, data, csv_writer):
      for StudentID in team:
          index = data.studentID.index(StudentID)
          csv_writer.writerow([
              tutorial_group,
              f"Team {team_number}",
              data.name[index],
              StudentID,
              data.gender[index],
              data.cgpa[index],
              data.school[index] ])

  progress_bar.description = "Charts Generated: Complete"
  progress_bar.bar_style = 'success'
  print("WORD Report (FCE1_Team3_YongDingAn.csv.docx) Completed")
  print("Start to generated the csv file...")
  with open('FCE1_Team5_YongDingAn.csv', 'w', newline='') as file:
      writer = csv.writer(file)
      writer.writerow(["TutorialGroup", "Team", "Name", "StudentID", "Gender", "CGPA", "School"])

      for group_name, data in data_set.items():
          team_counter = 1

          for team_tuple in data.teams:
              team_ids = team_tuple[0]
              TeamSummary_csv(group_name, team_counter, team_ids, data, writer)
              team_counter += 1

  print("CSV file (FCE1_Team3_YongDingAn.csv.csv) Completed")
  print("Program Finished!!! Pls view the result in the WORD doc and the CSV file.")

main has dropdown and button widgets.
Dropdown widget create a list of options to let user to choose the team size from 4 to 10.
The button widget to confirm the click event after user choose the team size.
After the button clicked, the rest of the program will run.


##Pseudocode for main
```
DECLARE Tutorial Group Size
DECLARE default Team Size is 5
CREATE dropdown = DropDown widget
			WITH options = [4, 5, 6, 7, 8, 9, 10]
			AND description = “choose team size”

CREATE button = Button widget
			WITH description = “submit”

DECLARE output = widget.Output()
FUNCTION on_button_clicked(INPUT button event)
			DECLARE global variable TEAM_SIZE
			BEGIN with output
					DECLARE output.clear_output()
					DECLARE TEAM_SIZE to dropdown value
					DECLARE data_set equal to FUNCTION CSV_reader
					DECLARE team_formation equal to FUNCTION create_team_formation(INPUT Tutorial Group Size)
					PRINT Start to generate best teams for each tutorial group with team size
					FOR data_set
							DECLARE best_teams to FUNCTION greedy_grouping(INPUT data, INPUT team_formation)
							DECLARE data.teams equal to best_teams
					END FOR
					PRINT All the best teams are generated with team size successfully

					FUNCTION output_csv_word(INPUT data)


			END with
LINK button.on_click(on_button_clicked)

DISPLAY (dropdown, button, output)
END FUNCTION

```

In [ ]:
if __name__ == "__main__":
    TUTORIAL_GROUP_SIZE = 50
    TEAM_SIZE = 5
    dropdown = widgets.Dropdown(
      options=[4, 5, 6, 7, 8, 9, 10],
      description='Choose Team Size (4-10):',
    )
    button = widgets.Button(description="Submit")
    output = widgets.Output()

    def on_button_clicked(b):
        global TEAM_SIZE
        with output:
            output.clear_output()
            TEAM_SIZE = dropdown.value
            data_set = CSV_reader()
            team_formation = create_team_formation(TUTORIAL_GROUP_SIZE)
            print(f"Start to generate best teams for each tutorial group with team size {TEAM_SIZE}...")
            for group, data in data_set.items():
                # print(f'Building best teams for {group}')
                best_teams = greedy_grouping(data, team_formation)
                data.teams = best_teams
            print("All the best teams are generated with team size successfully")

            # print(f"Final Teams and Details:\nMean CGPA:{data.mean_cgpa}")
            # print(f"'Males' in group: {data.gender.count('Male')}/{len(data.gender)}, \
            #       'Females' in group: {data.gender.count('Female')}/{len(data.gender)}")
            # for num, team in enumerate(best_teams):
            #     print_outcome(num, team[0], team[1], data)

            output_csv_word(data_set)

    button.on_click(on_button_clicked)
    display(dropdown, button, output)



NameError: name 'widgets' is not defined